# Replay saved baseline captions on Karpathy TEST5000

Attach mscoco-2014, select Accelerator None, enable Internet and Secret codex with access to tamvuu/cap-baseline/4. Alternatively, attach the supplied predictions file and set LOCAL_PREDICTIONS. Send tam_baseline_v4_test5000_cpu_review.zip.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tarfile, urllib.request

SOURCE_VERSION='tamvuu/cap-baseline/4'
SECRET_NAME='codex'
OUTPUT=Path('/kaggle/working/tam_baseline_v4_test5000_cpu')
def sha(path):
    with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def read(path):return json.loads(path.read_text(encoding='utf8'))
def save(path,value):path.write_text(json.dumps(value,indent=2,ensure_ascii=False,allow_nan=False)+'\n',encoding='utf8')
def download(url,path,digest):
    with urllib.request.urlopen(url,timeout=180) as src,path.open('wb') as dst:
        while chunk:=src.read(1024*1024):dst.write(chunk)
    if sha(path)!=digest:raise ValueError('Changed asset '+path.name)

COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
LOCAL_PREDICTIONS=''  # Optional mounted predictions.txt or predictions.json
EXPECTED_PREDICTIONS_SHA='56c7ba3ac3b931bc78f9002876c4b345c914846156cea3de29f6b920b68570fe'
EXPECTED_COCO_IDS_SHA='18a173e42d695050f9700423634f3caee4401a03ba9481320f57044806a8bc95'

if sha(COCO_JSON)!='2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d':
    raise ValueError('Wrong dataset_coco.json')
test=[r for r in read(COCO_JSON)['images'] if r['split']=='test']
by_imgid={int(r['imgid']):r for r in test}
ids=sorted(int(r['cocoid']) for r in test)
if len(test)!=5000 or len(by_imgid)!=5000 or len(set(ids))!=5000 or hashlib.sha256(json.dumps(ids,separators=(',',':')).encode()).hexdigest()!=EXPECTED_COCO_IDS_SHA:
    raise ValueError('Changed frozen TEST cohort')
if LOCAL_PREDICTIONS:
    predictions=Path(LOCAL_PREDICTIONS)
else:
    import json,os,re,subprocess,sys,tempfile,shutil
    from kaggle_secrets import UserSecretsClient
    try:token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    except Exception:raise RuntimeError('Enable the named Secret for the owner of Kaggle Output tamvuu/cap-baseline/4') from None
    if not token or token.startswith('sk-'):raise RuntimeError('Use a Kaggle API token in the named Secret')
    def run_cli(args,authenticated=True):
        environment=os.environ.copy()
        for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(k,None)
        if authenticated:environment['KAGGLE_API_TOKEN']=token
        environment['PYTHONUNBUFFERED']='1'
        process=subprocess.Popen(list(map(str,args)),env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace')
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        if process.wait():raise RuntimeError('API command failed; preserve staged files and read redacted log above')
    run_cli([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],authenticated=False)
    run_cli([sys.executable,'-u','-c',"import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as folder, patch.object(api,'build_kaggle_client') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output('ducanh3/notebook7bc83c4c76/89',path=folder)\n    req=endpoint.call_args.args[0]\n    assert (req.user_name,req.kernel_slug,req.version_label)==('ducanh3','notebook7bc83c4c76','v89')\n"],authenticated=False)
    
    download_root=Path(tempfile.mkdtemp(prefix='tam-baseline-v4-',dir='/tmp'))
    cli=[sys.executable,'-u','-c','from kaggle.cli import main; main()']
    run_cli([*cli,'kernels','output',SOURCE_VERSION,'-p',download_root,'--file-pattern',r'(^|/)artifacts/baseline/predictions[.]json$'])
    predictions,=download_root.rglob('predictions.json')

raw=json.loads(predictions.read_text(encoding='utf-8-sig'))
if len(raw)!=5000 or {r['imgid'] for r in raw}!=set(by_imgid) or any(not r['caption'].strip() for r in raw):
    raise ValueError('Prediction imgids do not exactly match frozen TEST')
canonical=json.dumps(sorted(raw,key=lambda r:r['imgid']),ensure_ascii=False,separators=(',',':')).encode()
if hashlib.sha256(canonical).hexdigest()!=EXPECTED_PREDICTIONS_SHA:
    raise ValueError('Saved v4 predictions differ from the supplied predictions.txt')
rows=sorted([dict(coco_id=int(by_imgid[r['imgid']]['cocoid']),caption=r['caption']) for r in raw],key=lambda r:r['coco_id'])
refs=sorted([dict(coco_id=int(r['cocoid']),references=[c['raw'] for c in r['sentences']]) for r in test],key=lambda r:r['coco_id'])
print('Matched all 5000 frozen Karpathy TEST images.',flush=True)
OUTPUT.mkdir()
deps=OUTPUT/'deps'; assets=OUTPUT/'assets';assets.mkdir()
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--target',str(deps),'pycocoevalcap==1.2','nltk==3.9.2','future','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],check=True)
sys.path.insert(0,str(deps))
os.environ['PYTHONPATH']=str(deps)+os.pathsep+os.environ.get('PYTHONPATH','')
base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
download(base+'chair.py',assets/'chair.py','98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d')
download(base+'chair.pkl',assets/'chair.pkl','73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3')
download('https://github.com/adoptium/temurin8-binaries/releases/download/jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz',assets/'java8.tar.gz','6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e')
with tarfile.open(assets/'java8.tar.gz') as archive:archive.extractall(assets/'jre',filter='data')
java,=(assets/'jre').rglob('bin/java')
os.environ['PATH']=str(java.parent)+os.pathsep+os.environ['PATH']
os.environ['JAVA_HOME']=str(java.parent.parent)
subprocess.run([sys.executable,'-c',"import nltk;assert nltk.download('punkt');assert nltk.download('punkt_tab')"],check=True)

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
tok=PTBTokenizer()
gts=tok.tokenize({r['coco_id']:[dict(caption=c) for c in r['references']] for r in refs})
hyp=tok.tokenize({r['coco_id']:[dict(caption=r['caption'])] for r in rows})
if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError('Incomplete tokenization')
cid,per=Cider().compute_score(gts,hyp)
bleu,_=Bleu(4).compute_score(gts,hyp)
rouge,_=Rouge().compute_score(gts,hyp)
save(OUTPUT/'cider_per_image.json',[dict(coco_id=i,CIDEr=float(v)) for i,v in zip(gts,per,strict=True)])
save(OUTPUT/'chair_input.json',[dict(image_id=r['coco_id'],caption=r['caption']) for r in rows])
subprocess.run([sys.executable,'-u',str(assets/'chair.py'),'--cap_file',str(OUTPUT/'chair_input.json'),'--cache',str(assets/'chair.pkl'),'--save_path',str(OUTPUT/'chair_output.json')],cwd=assets,check=True,timeout=600)
chair=read(OUTPUT/'chair_output.json')
sentences=chair['sentences'];captions={r['coco_id']:r['caption'] for r in rows}
if len(sentences)!=5000 or {r['image_id'] for r in sentences}!=set(captions) or any(r['caption']!=captions[r['image_id']] for r in sentences):
    raise ValueError('Wrong CHAIR caption coverage')

metrics=dict(CIDEr=float(cid),BLEU_1=float(bleu[0]),BLEU_4=float(bleu[3]),ROUGE_L=float(rouge))
metrics.update({k:chair['overall_metrics'][k] for k in ('CHAIRs','CHAIRi','Recall')})
result=dict(source=SOURCE_VERSION,predictions_sha256=EXPECTED_PREDICTIONS_SHA,images=5000,
            split='test',metrics=metrics,CIDEr_x100=100*metrics['CIDEr'])
save(OUTPUT/'result.json',result)
import zipfile
review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ('result.json','cider_per_image.json','chair_output.json'):
        z.write(OUTPUT/name,name)
print(json.dumps(result,indent=2))
print('SEND REVIEW:',review)
